# Stage 3: FF++ + SBI Fine-Tuning

Notebook này chạy riêng cho **Stage 3**.

Mục tiêu của Stage 3:

- load checkpoint đã train xong từ `Stage 2`
- train tiếp trên dữ liệu gồm:
  - `40%` real từ `FF++ train`
  - `45%` fake từ `FF++ train`
  - `15%` fake từ `SBI` sinh ra từ chính `FF++`
- test trên cả:
  - `FF++ test`
  - `Celeb-DF test`

Tức là Stage 3 hiện tại **không dùng Celeb-DF train**, chỉ dùng Celeb-DF để đánh giá OOD.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CS415_ROOT = '/content/drive/MyDrive/cs415'
DATA_ROOT = f'{CS415_ROOT}/data'
OUTPUT_ROOT = f'{CS415_ROOT}/outputs'
REPO_URL = 'https://github.com/kdnehihi/SBI-MoE-Deepfake-Detection.git'
PROJECT_DIR = '/content/SBI-MoE-Deepfake-Detection'

print('CS415_ROOT =', CS415_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)


In [ ]:
import os

%cd /content
if os.path.isdir(PROJECT_DIR):
    %cd {PROJECT_DIR}
    !git pull
else:
    !git clone "$REPO_URL" "$PROJECT_DIR"
    %cd {PROJECT_DIR}


In [ ]:
!pip install -q timm facenet-pytorch opencv-python==4.10.0.84 PyYAML Pillow scikit-learn


## Check Stage 2 Checkpoint

Stage 3 sẽ load từ `outputs/stage2_last.pt`, nên check trước cho chắc.


In [ ]:
!ls -lh "$OUTPUT_ROOT/stage2_last.pt"


## Build Stage 3 Dataset

Recipe hiện tại:

- `real = 0.40`
- `FF++ fake = 0.45`
- `SBI fake = 0.15`

Nếu bạn muốn giảm SBI xuống `10%`, chỉ cần sửa `--stage3-sbi-fake-ratio 0.10` và `--stage3-ffpp-fake-ratio 0.50`.


In [ ]:
!python data/prepare_stage_datasets.py \
  --stage stage3 \
  --celebdf-root "$DATA_ROOT/processed/celebdf" \
  --ffpp-root "$DATA_ROOT/processed/ffpp_generalization" \
  --output-root "$DATA_ROOT/stages/stage3_ffpp_sbi" \
  --val-ratio 0.125 \
  --stage3-real-ratio 0.40 \
  --stage3-ffpp-fake-ratio 0.45 \
  --stage3-sbi-fake-ratio 0.15 \
  --overwrite


In [ ]:
!ls -lh "$DATA_ROOT/stages/stage3_ffpp_sbi"/*.jsonl
!wc -l "$DATA_ROOT/stages/stage3_ffpp_sbi/train_manifest.jsonl"
!wc -l "$DATA_ROOT/stages/stage3_ffpp_sbi/val_manifest.jsonl"
!wc -l "$DATA_ROOT/stages/stage3_ffpp_sbi/test_ffpp_manifest.jsonl"
!wc -l "$DATA_ROOT/stages/stage3_ffpp_sbi/test_celebdf_manifest.jsonl"


## Train Stage 3

Stage 3 sẽ load checkpoint của Stage 2 và test cuối cùng trên cả `FF++` lẫn `Celeb-DF`.


In [ ]:
!python -u train_stage3.py \
  --dataset-root "$DATA_ROOT/stages/stage3_ffpp_sbi" \
  --init-checkpoint "$OUTPUT_ROOT/stage2_last.pt" \
  --output-dir "$OUTPUT_ROOT" \
  --batch-size 16 \
  --epochs 10 \
  --num-workers 2 \
  --image-size 224 \
  --device cuda


## Saved Outputs


In [ ]:
!ls -lh "$OUTPUT_ROOT"
